# PyDI Data Integration Workflow: Music

This notebook demonstrates comprehensive data integration using PyDI. We'll work with music datasets to showcase the data integration pipeline from entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Data Loading and Profiling](#part-1-data-loading-and-profiling)
- [Part 2: Entity Matching](#part-2-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
  - [Step 5: Machine Learning-based Matching Rules](#step-5-machine-learning-based-matching-rules)
- [Part 3: Data Fusion](#part-3-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input" / "music"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "music"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from utils import get_repo_root

ROOT = get_repo_root()
INPUT_DIR = ROOT / "usecases" / "input" / "music"
OUTPUT_DIR = ROOT / "usecases" / "output" / "music"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TypeError: unsupported operand type(s) for /: 'str' and 'str'

## Part 1: Data Loading and Profiling

In [ ]:
from PyDI.io import load_xml

# Load Discogs dataset
discogs = load_xml(
    INPUT_DIR / "data" / "discogs.xml",
    name="discogs",
    nested_handling="aggregate"
)

# Load MusicBrainz dataset
mbrainz = load_xml(
    INPUT_DIR / "data" / "musicbrainz.xml",
    name="musicbrainz",
    nested_handling="aggregate"
)

# Load Last.fm dataset
lastfm = load_xml(
    INPUT_DIR / "data" / "lastfm.xml",
    name="lastfm",
    nested_handling="aggregate"
)

# Display basic information
datasets = [discogs, mbrainz, lastfm]
names = ["Discogs", "MusicBrainz", "Last.fm"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 37,255


In [ ]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

### Attribute Coverage Analysis

In [ ]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,discogs_count,discogs_pct,discogs_coverage,discogs_samples,musicbrainz_count,musicbrainz_pct,musicbrainz_coverage,musicbrainz_samples,lastfm_count,lastfm_pct,lastfm_coverage,lastfm_samples,avg_coverage,max_coverage,datasets_with_attribute
0,artist,22627/22627,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",4763/4763,100.0%,1.000000,"['John B', 'Psychosis', 'Lypid']",9865/9865,100.0%,1.000000,"['John B', 'Psychosis', 'Petalpusher']",1.000000,1.000000,3
1,duration,22627/22627,100.0%,1.000000,"['0', '0', '0']",4763/4763,100.0%,1.000000,"['1055', '724', '2384']",4635/9865,47.0%,0.469843,"['903', '734', '1626']",0.823281,1.000000,3
2,genre,22627/22627,100.0%,1.000000,"['Electronic', 'Electronic', 'Electronic']",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.333333,1.000000,1
3,id,22627/22627,100.0%,1.000000,"['discogs_3', 'discogs_4', 'discogs_5']",4763/4763,100.0%,1.000000,"['mbrainz_1', 'mbrainz_2', 'mbrainz_3']",9865/9865,100.0%,1.000000,"['lastFM_1', 'lastFM_2', 'lastFM_4']",1.000000,1.000000,3
4,label,22627/22627,100.0%,1.000000,"['New Identity Recordings', 'Renegade Hardware...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.333333,1.000000,1
5,name,22627/22627,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",4763/4763,100.0%,1.000000,"['Fermats Theorem / Sight Beyond', 'Tempest / ...",9865/9865,100.0%,1.000000,"['John B - Fermats Theorem / Sight Beyond', '...",1.000000,1.000000,3
6,release-country,22027/22627,97.3%,0.973483,"['UK', 'UK', 'United States of America']",4015/4763,84.3%,0.842956,['United Kingdom of Great Britain and Northern...,0/0,0%,0.000000,N/A,0.605480,0.973483,2
7,release-date,20393/22627,90.1%,0.901268,"['1996-01-01', '1998-01-01', '2000-09-05']",4450/4763,93.4%,0.934285,"['1996-01-01', '1998-12-14', '2000-09-05']",0/0,0%,0.000000,N/A,0.611851,0.934285,2
8,tracks_track_duration,12894/22627,57.0%,0.569850,"[['405', '411', '420', '390'], ['108', '375', ...",4279/4763,89.8%,0.898383,"[['558', '497'], ['364', '360'], ['366', '284'...",4602/9865,46.6%,0.466498,"[['406', '497'], ['371', '363'], ['405', '411'...",0.644910,0.898383,3
9,tracks_track_name,22627/22627,100.0%,1.000000,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",4763/4763,100.0%,1.000000,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",4602/9865,46.6%,0.466498,"[['Fermats Theorem', 'Sight Beyond'], ['Tempes...",0.822166,1.000000,3



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['artist', 'duration', 'id', 'name', 'release-country', 'release-date', 'tracks_track_duration', 'tracks_track_name', 'tracks_track_position']


### Detailed Data Profiling

In [ ]:
from pathlib import Path

# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(datasets, names):
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")


Profiling Discogs...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 72.04it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles/discogs_profile.html
Profiling MusicBrainz...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 206.48it/s]


Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles/musicbrainz_profile.html
Profiling Last.fm...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 175.66it/s]

Profile saved: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles/lastfm_profile.html

 Generated 3 detailed HTML reports
 Location: /Users/luca/PycharmProjects/PyDI/usecases/output/music/dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • discogs_profile.html
  • musicbrainz_profile.html
  • lastfm_profile.html


## Part 2: Entity Matching

### Step 1: Blocking

In [ ]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [ ]:
# Import blocking methods
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - Longest Token in Name
# Add name_longest_token directly to the original dataframes
def get_longest_token(name):
    tokens = re.split(r"[^A-Za-z0-9_']+", str(name))    
    tokens = [t for t in tokens if t]
    return max(tokens, key=len) if tokens else ''

mbrainz['name_longest_token'] = mbrainz['name'].apply(get_longest_token)
discogs['name_longest_token'] = discogs['name'].apply(get_longest_token)

standard_blocker_m2d = StandardBlocker(
    mbrainz, discogs,
    on=['name_longest_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
 )
standard_candidates_m2d = standard_blocker_m2d.materialize()

sn_blocker_m2d = SortedNeighbourhoodBlocker(
    mbrainz, discogs,
    key='name',  # Sort by name
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
sn_candidates_m2d = sn_blocker_m2d.materialize()

token_blocker_m2d = TokenBlocker(
    mbrainz, discogs,
    column='name',      # Tokenize names
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id',
    ngram_size=5,
    ngram_type='character'
)
token_candidates_m2d = token_blocker_m2d.materialize()

embedding_blocker_m2d = EmbeddingBlocker(
    mbrainz, discogs,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_m2d = embedding_blocker_m2d.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4685 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1443 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 27390 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/debugR

### Step 2: Evaluate Blocking Against Ground Truth

In [ ]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_m2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 10 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 13 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 25 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 42 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 48 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 55 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 60 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 63 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 79 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 98 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 110 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 131 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 137 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 141 true matches
[INFO ]

{'pair_completeness': 1.0,
 'pair_quality': 0.002145844289888658,
 'reduction_ratio': 0.9967007230357613,
 'total_candidates': 355571,
 'total_possible_pairs': 107772401,
 'true_positives_found': 763,
 'total_true_pairs': 763,
 'batches_processed': 356,
 'evaluation_timestamp': '2025-10-28T15:08:56.401673',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/blocking_detailed_results.csv']}

In [ ]:
lastfm['name_longest_token'] = lastfm['name'].apply(get_longest_token)

standard_blocker_m2l = StandardBlocker(
    mbrainz, lastfm,
    on=['name_longest_token'],  # Block on longest token in name
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_m2l = standard_blocker_m2l.materialize()

sn_blocker_m2l = SortedNeighbourhoodBlocker(
    mbrainz, lastfm,
    key='name',  # Sort by name
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
sn_candidates_m2l = sn_blocker_m2l.materialize()

token_blocker_m2l = TokenBlocker(
    mbrainz, lastfm,
    column='name',      # Tokenize names
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id',
    ngram_size=5,
    ngram_type='character'
)
token_candidates_m2l = token_blocker_m2l.materialize()

embedding_blocker_m2l = EmbeddingBlocker(
    mbrainz, lastfm,
    text_cols=['name'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
embedding_candidates_m2l = embedding_blocker_m2l.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1720 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 2850 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1368 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created sorted neighbourhood with window size 20
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - created 1 sorted sequence from 14628 records
[INFO ] PyDI.entitymatching.blocking.sorted_neighbourhood.SortedNeighbourhoodBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/music/blocking-evaluation/debugR

Now let's evaluate which blocking method we want to use for each dataset combination:

In [ ]:
# Evaluate all blocking methods for both dataset combinations
evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
m2d_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2d, standard_blocker_m2d],
    'SortedNeighbourhoodBlocker': [sn_candidates_m2d, sn_blocker_m2d],
    'TokenBlocking': [token_candidates_m2d, token_blocker_m2d],
    'EmbeddingBlocking': [embedding_candidates_m2d, embedding_blocker_m2d]
}

# Load correspondences for evaluation
m2d_correspondences = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_test.csv",
    name="m2d_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Evaluate blocking for m2d datasets
m2d_results = []
for method_name, candidates in m2d_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], m2d_correspondences, candidates[1], out_dir=OUTPUT_DIR / "blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'm2d'
    m2d_results.append(result)

# Select best method for each dataset (highest pair_completeness, then highest reduction_ratio)
m2d_best = max(m2d_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))

print(f"Best blocking for m2d: {m2d_best['method']} (PC: {m2d_best['pair_completeness']:.3f}, RR: {m2d_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 1.000
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   0.997
[INFO ] root -   True Matches Found: 475/475
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.994
[INFO ] root -   Pair Quality:      0.003
[INFO ] root -   Reduction Ratio:   0.999
[INFO ] root -   True Matches Found: 472/475
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 1.000
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   0.981
[INFO ] root -   True Matches Found: 475/475
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.989
[INFO ] root -   Pair Quality:      0.005
[INFO ] root -   Reduction Ratio:   0.999
[INFO ] root -   True Matches Found: 470/475
[INFO ] root - Blocking evaluation complete!


Best blocking for m2d: StandardBlocking (PC: 1.000, RR: 0.997)


In [ ]:
m2l_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2l, standard_blocker_m2l],
    'SortedNeighbourhood': [sn_candidates_m2l, sn_blocker_m2l],
    'TokenBlocking': [token_candidates_m2l, token_blocker_m2l],
    'EmbeddingBlocking': [embedding_candidates_m2l, embedding_blocker_m2l]
}

m2l_correspondences = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_test.csv",
    name="m2l_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Evaluate blocking for m2l datasets
m2l_results = []
for method_name, candidates in m2l_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], m2l_correspondences,candidates[1], out_dir=OUTPUT_DIR / "blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'm2l'
    m2l_results.append(result)

m2l_best = max(m2l_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))

print(f"Best blocking for m2l: {m2l_best['method']} (PC: {m2l_best['pair_completeness']:.3f}, RR: {m2l_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 0.938
[INFO ] root -   Pair Quality:      0.003
[INFO ] root -   Reduction Ratio:   0.997
[INFO ] root -   True Matches Found: 485/517
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.696
[INFO ] root -   Pair Quality:      0.003
[INFO ] root -   Reduction Ratio:   0.997
[INFO ] root -   True Matches Found: 360/517
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.998
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   0.981
[INFO ] root -   True Matches Found: 516/517
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.938
[INFO ] root -   Pair Quality:      0.005
[INFO ] root -   Reduction Ratio:   0.998
[INFO ] root -   True Matches Found: 485/517
[INFO ] root - Blocking evaluation complete!


Best blocking for m2l: TokenBlocking (PC: 0.998, RR: 0.981)


### Step 3: Entity Matching with Comparators

In [ ]:
from PyDI.entitymatching import StringComparator, DateComparator, NumericComparator

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators = [
    # Release name — Jaccard
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release artist — Jaccard
    StringComparator(
        column='artist',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release date — within 2 years
    DateComparator(
        column='release-date',
        max_days_difference=365 * 2
    ),
    # Release country — Jaccard
    StringComparator(
        column='release-country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    # Release duration — within 10% --> allow 10% deviation
    NumericComparator(
        column='duration',
        method='relative_difference',
        max_difference=0.10
    ),
    # # Track list — overlap
    StringComparator(
        column='tracks_track_name',
        similarity_function='jaccard',
        preprocess=normalize_text,
        list_strategy="set_overlap"
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [ ]:
import numpy as np

# Convert lists in mbrainz["duration"] to single integer values (sum if list, else int)

def sum_duration(val):
    if isinstance(val, list):
        return int(np.nansum([int(x) for x in val if str(x).isdigit()]))
    try:
        return int(val)
    except Exception:
        return np.nan

mbrainz["duration"] = mbrainz["duration"].apply(sum_duration)

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_m2d = matcher.match(
    df_left=mbrainz,
    df_right=discogs, 
    candidates=standard_blocker_m2d, # pass the blocker, which will internally generate candidate pairs using batching
    comparators=comparators,
    weights=None, # equal weights for all 4 comparators
    threshold=0.5,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 4763 x 22627 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 4763 x 22627 elements after 0:00:0.377; 355571 blocked pairs (reduction ratio: 0.9967007230357613)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:472.285; found 4174 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [ ]:
gt_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_discogs_test.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  473
[INFO ] root -   True Negatives:  5095
[INFO ] root -   False Positives: 235
[INFO ] root -   False Negatives: 2
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.959
[INFO ] root -   Precision: 0.668
[INFO ] root -   Recall:    0.996
[INFO ] root -   F1-Score:  0.800


{'precision': 0.6680790960451978,
 'recall': 0.9957894736842106,
 'f1': 0.7996618765849536,
 'accuracy': 0.9591731266149871,
 'true_positives': 473,
 'false_positives': 235,
 'false_negatives': 2,
 'true_negatives': 5095,
 'threshold_used': 0.0,
 'total_correspondences': 4174,
 'filtered_correspondences': 4174,
 'evaluation_timestamp': '2025-10-28T15:24:31.492882',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_detailed_results.csv']}

In [ ]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

Analyzing cluster size distribution in our entity matching results...


[INFO ] root - Cluster Size Distribution of 2968 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	2352	|	79.25%
[INFO ] root - 		3	|	374	|	12.60%
[INFO ] root - 		4	|	131	|	4.41%
[INFO ] root - 		5	|	48	|	1.62%
[INFO ] root - 		6	|	29	|	0.98%
[INFO ] root - 		7	|	9	|	0.30%
[INFO ] root - 		8	|	10	|	0.34%
[INFO ] root - 		9	|	3	|	0.10%
[INFO ] root - 		10	|	7	|	0.24%
[INFO ] root - 		12	|	3	|	0.10%
[INFO ] root - 		14	|	1	|	0.03%
[INFO ] root - 		18	|	1	|	0.03%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/music/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,2352,79.245283
1,3,374,12.601078
2,4,131,4.413747
3,5,48,1.617251
4,6,29,0.977089
5,7,9,0.303235
6,8,10,0.336927
7,9,3,0.101078
8,10,7,0.235849
9,12,3,0.101078


In [ ]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_m2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/music/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 2968 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [ ]:
from PyDI.entitymatching import MaximumBipartiteMatching

# use Maximum Bipartite Matching to refine results to 1:1 matches
clusterer = MaximumBipartiteMatching()
correspondences_m2d = clusterer.cluster(correspondences_m2d)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2d,
    test_pairs=gt_test,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Filtered correspondences: 4174 -> 4174 (threshold=0.0)
[INFO ] root - Maximum bipartite matching: 4174 -> 3089 
[INFO ] root - MaximumBipartiteMatching: 4174 -> 3089 correspondences
[INFO ] root - MaximumBipartiteMatching: 7072 -> 6178 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  439
[INFO ] root -   True Negatives:  5248
[INFO ] root -   False Positives: 82
[INFO ] root -   False Negatives: 36
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.980
[INFO ] root -   Precision: 0.843
[INFO ] root -   Recall:    0.924
[INFO ] root -   F1-Score:  0.882


{'precision': 0.8426103646833013,
 'recall': 0.9242105263157895,
 'f1': 0.8815261044176707,
 'accuracy': 0.9796726959517658,
 'true_positives': 439,
 'false_positives': 82,
 'false_negatives': 36,
 'true_negatives': 5248,
 'threshold_used': 0.0,
 'total_correspondences': 3089,
 'filtered_correspondences': 3089,
 'evaluation_timestamp': '2025-10-28T15:24:34.331808',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_detailed_results.csv']}

### Step 5: Machine Learning-based Matching Rules

In [ ]:
from PyDI.entitymatching import FeatureExtractor

# Load ground truth correspondences
m2l_train = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_train_small.csv",
    name="ground_truth_train",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

m2l_test = load_csv(
    INPUT_DIR / "entitymatching" / "musicbrainz_2_lastfm_test.csv",
    name="ground_truth_test",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

similarity_comparators = [
    # Name similarity features - most important for movie matching
    StringComparator("name", similarity_function="jaccard", preprocess=normalize_text),
    StringComparator("name", similarity_function="levenshtein", preprocess=normalize_text),
    StringComparator("name", similarity_function="cosine", preprocess=normalize_text),

    # Artist name similarity
    StringComparator("artist", similarity_function="jaccard", preprocess=normalize_text),
    StringComparator("artist", similarity_function="levenshtein", preprocess=normalize_text),

    StringComparator("tracks_track_name", similarity_function="jaccard", preprocess=normalize_text, list_strategy="set_overlap"),
    NumericComparator("duration", method="relative_difference", max_difference=0.10),
]

feature_extractor = FeatureExtractor(similarity_comparators)

# Extract features using FeatureExtractor
train_features = feature_extractor.create_features(
    mbrainz, lastfm, m2l_train[['id1', 'id2']], labels=m2l_train['label'], id_column='id'
)

print(f"✅ Training features extracted!")
print(f"Feature columns: {[col for col in train_features.columns if col not in ['id1', 'id2', 'label']]}")

# Prepare data for ML training
feature_columns = [col for col in train_features.columns if col not in ['id1', 'id2', 'label']]

X_train = train_features[feature_columns]
y_train = train_features['label']

print(f"Training data: X={X_train.shape}, y={y_train.shape}")
print(f"Class distribution: {y_train.value_counts().to_dict()}")

[INFO ] root - Label distribution: 1702 positive, 15790 negative


✅ Training features extracted!
Feature columns: ['StringComparator(name, jaccard, tokenization=word, list_strategy=None)', 'StringComparator(name, levenshtein, tokenization=char, list_strategy=None)', 'StringComparator(name, cosine, tokenization=word, list_strategy=None)', 'StringComparator(artist, jaccard, tokenization=word, list_strategy=concatenate)', 'StringComparator(artist, levenshtein, tokenization=char, list_strategy=best_match)', 'StringComparator(tracks_track_name, jaccard, tokenization=word, list_strategy=set_overlap)', 'NumericComparator(duration, relative_difference, list_strategy=None)']
Training data: X=(17492, 7), y=(17492,)
Class distribution: {False: 15790, True: 1702}


#### Full Scikit-learn integration

In [ ]:
# Set up GridSearchCV with multiple models and hyperparameters
print(f"\n🔍 Setting up GridSearchCV...")

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(X_train, y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")


🔍 Setting up GridSearchCV...
GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

  ✅ RandomForest: Best CV F1 = 0.9715
     Best params: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.9567
     Best params: {'C': 10.0, 'class_weight': None, 'penalty': 'l2'}

Training GradientBoosting...
  ✅ GradientBoosting: Best CV F1 = 0.9682
     Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}

Training SVM...
  ✅ SVM: Best CV F1 = 0.9690
     Best params: {'C': 10.0, 'class_weight': None, 'kernel': 'rbf'}

🏆 Best Overall Model: RandomForest (CV F1: 0.9715)


Now, we can directly use the trained model with PyDIs MLBasedMatcher

In [ ]:
from PyDI.entitymatching import MLBasedMatcher

# Create MLBasedMatcher and apply trained model
ml_matcher = MLBasedMatcher(feature_extractor)

correspondences_m2l = ml_matcher.match(
    mbrainz, lastfm, candidates=token_blocker_m2l, id_column='id', trained_classifier=best_overall_model
)

[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 4763 x 9865 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 4763 x 9865 elements after 0:00:2.513; 907602 blocked pairs (reduction ratio: 0.9806839743635446)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:568.282; found 3259 correspondences.


In [ ]:
# Show feature importance if available
if hasattr(best_overall_model, 'feature_importances_'):
    print(f"\n🔍 Top Feature Importances:")
    importance_df = ml_matcher.get_feature_importance(best_overall_model, feature_columns)
    display(importance_df.head(8))


🔍 Top Feature Importances:


,feature,importance
3,"StringComparator(artist, jaccard, tokenization...",0.3232
5,"StringComparator(tracks_track_name, jaccard, t...",0.2896
4,"StringComparator(artist, levenshtein, tokeniza...",0.1580
6,"NumericComparator(duration, relative_differenc...",0.1360
2,"StringComparator(name, cosine, tokenization=wo...",0.0559
0,"StringComparator(name, jaccard, tokenization=w...",0.0275
1,"StringComparator(name, levenshtein, tokenizati...",0.0098


In [ ]:
eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_m2l,
    test_pairs=m2l_test,
    out_dir=debug_output_dir
)

display(eval_results)

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_m2l,
    out_dir=OUTPUT_DIR / "cluster_analysis"
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  505
[INFO ] root -   True Negatives:  4720
[INFO ] root -   False Positives: 12
[INFO ] root -   False Negatives: 12
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.995
[INFO ] root -   Precision: 0.977
[INFO ] root -   Recall:    0.977
[INFO ] root -   F1-Score:  0.977


{'precision': 0.97678916827853,
 'recall': 0.97678916827853,
 'f1': 0.97678916827853,
 'accuracy': 0.9954277005143837,
 'true_positives': 505,
 'false_positives': 12,
 'false_negatives': 12,
 'true_negatives': 4720,
 'threshold_used': 0.0,
 'total_correspondences': 3259,
 'filtered_correspondences': 3259,
 'evaluation_timestamp': '2025-10-28T15:35:21.334537',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/music/debug_results_entity_matching/matching_detailed_results.csv']}

[INFO ] root - Cluster Size Distribution of 3145 clusters:
[INFO ] root - 	Cluster Size	| Frequency	| Percentage
[INFO ] root - 	──────────────────────────────────────────────────
[INFO ] root - 		2	|	3080	|	97.93%
[INFO ] root - 		3	|	39	|	1.24%
[INFO ] root - 		4	|	20	|	0.64%
[INFO ] root - 		5	|	1	|	0.03%
[INFO ] root - 		6	|	4	|	0.13%
[INFO ] root - 		7	|	1	|	0.03%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/music/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,3080,97.933227
1,3,39,1.240064
2,4,20,0.635930
3,5,1,0.031797
4,6,4,0.127186
5,7,1,0.031797


## Part 3: Data Fusion

In [ ]:
mbrainz["mbrainz_id"] = mbrainz["id"]

# Assign trust scores to datasets
mbrainz.attrs["trust_score"] = 1
discogs.attrs["trust_score"] = 2
lastfm.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_m2d, correspondences_m2l], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 6,348


## Step 1: Define Fusion Strategy 

In [ ]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, voting, maximum

strategy = DataFusionStrategy('music_fusion_strategy')

strategy.add_attribute_fuser('name', shortest_string)
strategy.add_attribute_fuser('artist', longest_string)
strategy.add_attribute_fuser('release-date', voting)
strategy.add_attribute_fuser('release-country', longest_string)
strategy.add_attribute_fuser('duration', maximum)
strategy.add_attribute_fuser('tracks_track_name', union)
strategy.add_attribute_fuser('label', longest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'shortest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'artist' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-date' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'release-country' using rule 'longest_string'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'duration' using rule 'maximum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'tracks_track_name' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'label' using rule 'longest_string'


## Step 2: Run Fusion

In [ ]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[mbrainz, discogs, lastfm],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/music/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'music_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 10853 of 10853 unique IDs
[INFO ] PyDI.fusion.engine - Created 30918 record groups from 6348 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 30918 groups:
[INFO ] PyDI.fusion.engine -     Group Size | Frequency
[INFO ] PyDI.fusion.engine -     ----------------------
[INFO ] PyDI.fusion.engine -           1 |   26402
[INFO ] PyDI.fusion.engine -           2 |    2808
[INFO ] PyDI.fusion.engine -           3 |    1648
[INFO ] PyDI.fusion.engine -           4 |      31
[INFO ] PyDI.fusion.engine -           5 |      17
[INFO ] PyDI.fusion.engine -           6 |       6
[INFO ] PyDI

Fused rows: 4,516


,_id,_fusion_group_id,_fusion_sources,tracks_track_name,id,tracks_track_position,tracks_track_duration,release-date,name_longest_token,release-country,mbrainz_id,artist,name,duration,_fusion_confidence,_fusion_metadata,label,genre
0,lastFM_14765,group_0,"[musicbrainz, lastfm]",[Six Fantasies on a Poem by Thomas Campion: He...,lastFM_14765,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]","[171, 225, 243, 186, 360, 150, 429, 276, 476, ...",1994-01-01,Fantasies,United States of America,mbrainz_6958,"Lansky, Paul",Fantasies & Tableaux,2850.0,0.654167,"{'tracks_track_name_rule': 'union', 'tracks_tr...",NaN,NaN
1,lastFM_3437,group_1,"[musicbrainz, lastfm, discogs]","[Clearing, Deep In The Grass, Deep in the Gras...",lastFM_3437,"[1, 2, 3, 4, 5, 6, 7]","[639, 623, 601, 295, 768, 394, 438]",1997-01-01,Mercury,United States of America,mbrainz_1473,"Stokes, Saul",Washed In Mercury,3758.0,0.586806,"{'tracks_track_name_rule': 'union', 'tracks_tr...",Hypnos,Electronic
2,discogs_33211,group_2,"[discogs, lastfm, musicbrainz]","[Bach to Back, Bahaha Hahi, Dexter, Easy Lee, ...",discogs_33211,"[1, 2, 3, 4, 5, 6, 7, 8, 9]","[606, 496, 455, 561, 490, 559, 482, 545, 462]",2003-09-19,Alcachofa,Germany,mbrainz_5933,"Villalobos, Ricardo",Alcachofa,4656.0,0.602193,"{'tracks_track_name_rule': 'union', 'tracks_tr...",Playhouse,Electronic
3,mbrainz_16252,group_3,"[discogs, musicbrainz]","[Clouded, Earthly Love, Faceless Ones, Illumin...",mbrainz_16252,"[1, 2, 3, 4, 5, 6, 1, 2, 3, 4, 5, 6]","[244, 244, 187, 370, 260, 572, 205, 327, 208, ...",2012-01-01,Obscura,Canada,mbrainz_16252,Gorguts,Obscura,3658.0,0.583333,"{'tracks_track_name_rule': 'union', 'tracks_tr...",War On Music,Rock
4,discogs_100640,group_4,"[discogs, musicbrainz]","[""Gluttony"" Lyric Video, ""Gluttony"" Music Vide...",discogs_100640,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...","[161, 233, 192, 245, 262, 278, 233, 254, 243, ...",2013-02-15,Confessions,United States of America,mbrainz_21414,Buckcherry,Confessions,3457.0,0.625000,"{'tracks_track_name_rule': 'union', 'tracks_tr...",Century Media Records,Rock


## Step 3: Evaluate Data Fusion

In [ ]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("artist", tokenized_match)
strategy.add_evaluation_function("duration", numeric_tolerance_match)
strategy.add_evaluation_function("release-date", year_only_match)
strategy.add_evaluation_function("release-country", tokenized_match)
strategy.add_evaluation_function("label", tokenized_match)
strategy.add_evaluation_function("tracks_track_name", set_equality_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'artist'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'duration'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-date'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'release-country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'label'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'tracks_track_name'


In [ ]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='mbrainz_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/music/data_fusion/debug_fusion_eval.jsonl for mismatch details.


[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.766 overall accuracy (151/197)


Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.766
  macro_accuracy: 0.764
  num_evaluated_records: 23
  num_evaluated_attributes: 9
  total_evaluations: 197
  total_correct: 151
  tracks_track_name_accuracy: 0.130
  tracks_track_name_count: 23
  tracks_track_position_accuracy: 0.696
  tracks_track_position_count: 23
  release-date_accuracy: 1.000
  release-date_count: 23
  tracks_track_duration_accuracy: 0.500
  tracks_track_duration_count: 20
  release-country_accuracy: 0.957
  release-country_count: 23
  label_accuracy: 0.812
  label_count: 16
  artist_accuracy: 0.913
  artist_count: 23
  name_accuracy: 0.913
  name_count: 23
  duration_accuracy: 0.957
  duration_count: 23

Overall Accuracy: 76.6%
